# Demo End-to-End: Fases 1 a 6

Este notebook ejecuta la solución completa de `Phase 1` a `Phase 6` y deja una lectura visual por fase para presentación.

Estructura por fase:
1. explicación breve de qué se va a hacer,
2. ejecución del script,
3. resumen visual de los artefactos y métricas más importantes.

La `Phase 5` y la `Phase 6` usan exactamente los comandos que has fijado para el experimento PPO con selección programática de clusters temporales.

In [3]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='talk')
except Exception:
    sns = None
    plt.style.use('ggplot')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    raise RuntimeError('Ejecuta este notebook desde la raiz del proyecto.')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)


def run_python(args: list[str]) -> None:
      cmd = [sys.executable, *args]
      print('> ' + ' '.join(shlex.quote(x) for x in cmd))

      process = subprocess.Popen(
          cmd,
          cwd=PROJECT_ROOT,
          stdout=subprocess.PIPE,
          stderr=subprocess.STDOUT,
          text=True,
          bufsize=1,
      )

      assert process.stdout is not None
      for line in process.stdout:
          print(line, end='')

      return_code = process.wait()
      if return_code != 0:
          raise subprocess.CalledProcessError(return_code, cmd)


def latest_run_dir(root: Path) -> Path:
    latest_txt = root / 'latest_run.txt'
    if latest_txt.exists():
        run_id = latest_txt.read_text(encoding='utf-8').strip()
        candidate = root / run_id
        if candidate.exists():
            return candidate
    candidates = [p for p in root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No runs found under {root}')
    return max(candidates, key=lambda p: p.stat().st_mtime)


def latest_matching_dir(root: Path, needle: str) -> Path:
    candidates = [p for p in root.iterdir() if p.is_dir() and needle in p.name]
    if not candidates:
        raise FileNotFoundError(f'No run under {root} matches {needle!r}')
    return max(candidates, key=lambda p: p.stat().st_mtime)


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def tidy_index_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if 'Unnamed: 0' in df.columns:
        df = df.rename(columns={'Unnamed: 0': 'index_key'})
    return df


## Phase 1: Data Loading & Feature Engineering

En esta fase construimos el dataset procesado que alimenta todo el pipeline. El objetivo es cargar el benchmark y los algoritmos, generar la matriz de retornos, calcular estadísticas básicas e inferir exposición subyacente por clase de activo y dirección. Después resumiremos: evolución del benchmark, métricas principales, distribución de activos inferidos y nivel de confianza de la inferencia.

In [4]:
run_python(['scripts/run_phase1.py'])

00:15:02 | INFO     | 
00:15:02 | INFO     | PHASE COMPLETE: Phase 1: Data Loading & Feature Engineering
00:15:02 | INFO     | ======================================================================
00:15:02 | INFO     | Total time: 17m 45s
00:15:02 | INFO     | Memory delta: +523.6 MB
00:15:02 | INFO     | Steps: 5/5 completed
00:15:02 | INFO     | Metrics saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\data_pipeline\20260327_235717\phase1_metrics.json
00:15:02 | INFO     | Results saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\data_pipeline\20260327_235717\phase1_results.json
00:15:02 | INFO     | 
00:15:02 | INFO     | PHASE COMPLETE: Phase 1: Data Loading & Feature Engineering
00:15:02 | INFO     | ======================================================================
00:15:02 | INFO     | Total time: 17m 45s
00:15:02 | INFO     | Memory delta: +524.2 MB
00:15:02 | INFO     | Steps: 5/5 completed

PHASE SUMMARY: Phase 1: Data Loading & 

In [5]:
phase1_dir = latest_run_dir(PROJECT_ROOT / 'outputs' / 'data_pipeline')
phase1 = read_json(phase1_dir / 'phase1_results.json')
benchmark_metrics_path = PROJECT_ROOT / 'data' / 'processed' / 'analysis' / 'benchmark_profile' / 'metrics.json'
benchmark_metrics = read_json(benchmark_metrics_path) if benchmark_metrics_path.exists() else {}
asset_inf = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'algorithms' / 'asset_inference.csv')
bench = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'benchmark' / 'daily_returns.csv')
bench = bench.rename(columns={'Unnamed: 0': 'date'})
bench['date'] = pd.to_datetime(bench['date'], errors='coerce')
bench['benchmark'] = pd.to_numeric(bench['benchmark'], errors='coerce').fillna(0.0)
bench['equity_curve'] = (1.0 + bench['benchmark']).cumprod()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
axes[0].plot(bench['date'], bench['equity_curve'], color='black', lw=2)
axes[0].set_title('Benchmark Equity Curve')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Growth of $1')

asset_counts = asset_inf['asset_class'].fillna('unknown').value_counts().sort_values(ascending=False)
axes[1].bar(asset_counts.index.astype(str), asset_counts.values, color='#4C78A8')
axes[1].set_title('Distribucion de Activos Inferidos')
axes[1].tick_params(axis='x', rotation=45)

conf = pd.to_numeric(asset_inf['confidence'], errors='coerce').fillna(0.0)
axes[2].hist(conf, bins=20, color='#F58518', alpha=0.85)
axes[2].axvline(conf.mean(), color='black', linestyle='--', label=f"media={conf.mean():.2f}")
axes[2].set_title('Confianza de Inferencia')
axes[2].set_xlabel('confidence')
axes[2].legend()
plt.tight_layout()
plt.show()

metrics_table = pd.DataFrame([{
    'algoritmos_encontrados': phase1.get('n_algorithms_found'),
    'algoritmos_procesados': phase1.get('processing', {}).get('n_processed'),
    'retorno_benchmark_anual': benchmark_metrics.get('annualized_return'),
    'vol_benchmark_anual': benchmark_metrics.get('annualized_volatility'),
    'sharpe_benchmark': benchmark_metrics.get('sharpe_ratio'),
    'max_drawdown_benchmark': benchmark_metrics.get('max_drawdown'),
    'inferencia_media_confianza': phase1.get('inference', {}).get('mean_confidence'),
}])
display(metrics_table)

display(asset_inf.groupby('asset_class', dropna=False).agg(
    n_algorithms=('algo_id', 'count'),
    mean_confidence=('confidence', 'mean'),
    median_confidence=('confidence', 'median')
).sort_values('n_algorithms', ascending=False))

,n_algorithms,mean_confidence,median_confidence
asset_class,,,
mixed,5394,29.452725,26.0
unknown,3906,0.024322,0.0
equity,3412,33.759086,30.0
forex,492,59.770325,67.0
indices,126,54.857143,48.5
commodities,111,47.810811,39.0
etf,65,54.600000,48.0
crypto,4,33.250000,33.5
rates,3,69.666667,67.0


## Phase 2: Analysis & Reverse Engineering

Aquí caracterizamos el universo y construimos la capa semántica del problema: perfiles por algoritmo, clustering conductual, clustering temporal y regímenes. La idea para presentación es enseñar que el universo no se trata como una caja negra homogénea, sino como una colección de familias y clusters con rasgos estadísticos distintos. El resumen visual mostrará tamaños de familia y un scatter retorno-volatilidad coloreado por familia, además de una tabla con los clusters temporales acumulados más relevantes del último corte disponible.

In [6]:
run_python(['scripts/run_phase2.py'])

00:30:17 | INFO     | 
00:30:17 | INFO     | PHASE COMPLETE: Phase 2: Analysis & Reverse Engineering
00:30:17 | INFO     | ======================================================================
00:30:17 | INFO     | Total time: 10m 35s
00:30:17 | INFO     | Memory delta: +614.4 MB
00:30:17 | INFO     | Steps: 7/7 completed
00:30:17 | INFO     | Metrics saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\analysis\20260328_001942_gmm_direct_r4\phase2_metrics.json
00:30:17 | INFO     | Results saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\analysis\20260328_001942_gmm_direct_r4\phase2_results.json
00:30:17 | INFO     | 
00:30:17 | INFO     | PHASE COMPLETE: Phase 2: Analysis & Reverse Engineering
00:30:17 | INFO     | ======================================================================
00:30:17 | INFO     | Total time: 10m 35s
00:30:17 | INFO     | Memory delta: +614.9 MB
00:30:17 | INFO     | Steps: 7/7 completed

PHASE SUMMARY: Phase 2: Analy

In [7]:
behavioral = tidy_index_csv(PROJECT_ROOT / 'data' / 'processed' / 'analysis' / 'clustering' / 'behavioral' / 'features.csv')
family_labels = tidy_index_csv(PROJECT_ROOT / 'data' / 'processed' / 'analysis' / 'clustering' / 'behavioral' / 'family_labels.csv')
cluster_history = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'analysis' / 'clustering' / 'temporal' / 'cluster_history.csv')

behavioral = behavioral.rename(columns={'index_key': 'algo_id'})
family_labels = family_labels.rename(columns={'index_key': 'algo_id'})
family_df = behavioral.merge(family_labels[['algo_id', 'family']], on='algo_id', how='left')
family_sizes = family_df['family'].value_counts(dropna=False).head(12)

valid_points = family_df[['ann_vol', 'ann_return', 'family']].dropna().copy()
valid_points = valid_points.sample(min(len(valid_points), 3000), random_state=42)

latest_week = pd.to_datetime(cluster_history['week_end'], errors='coerce').max()
latest_clusters = cluster_history[pd.to_datetime(cluster_history['week_end'], errors='coerce') == latest_week].copy()
latest_clusters = latest_clusters[latest_clusters['cluster_cumulative'].astype(str) != 'insufficient_data']
latest_clusters['cluster_cumulative'] = latest_clusters['cluster_cumulative'].astype(str)
cluster_summary = latest_clusters.merge(
    behavioral[['algo_id', 'ann_return', 'ann_vol', 'sharpe', 'sortino']],
    on='algo_id',
    how='left'
).groupby('cluster_cumulative').agg(
    n_algorithms=('algo_id', 'count'),
    ann_return_mean=('ann_return', 'mean'),
    ann_vol_mean=('ann_vol', 'mean'),
    sharpe_mean=('sharpe', 'mean'),
    sortino_mean=('sortino', 'mean')
).sort_values(['sharpe_mean', 'ann_return_mean'], ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
axes[0].bar(family_sizes.index.astype(str), family_sizes.values, color='#72B7B2')
axes[0].set_title('Tamano de Familias Conductuales')
axes[0].set_xlabel('family')
axes[0].set_ylabel('n algoritmos')

axes[1].scatter(
    valid_points['ann_vol'],
    valid_points['ann_return'],
    c=pd.factorize(valid_points['family'].astype(str))[0],
    cmap='tab20',
    alpha=0.65,
    s=25,
)
axes[1].set_title('Retorno vs Volatilidad por Familia')
axes[1].set_xlabel('annualized volatility')
axes[1].set_ylabel('annualized return')
plt.tight_layout()
plt.show()

display(cluster_summary.head(10).style.format({
    'ann_return_mean': '{:.2%}',
    'ann_vol_mean': '{:.2%}',
    'sharpe_mean': '{:.2f}',
    'sortino_mean': '{:.2f}',
}))

,n_algorithms,ann_return_mean,ann_vol_mean,sharpe_mean,sortino_mean
cluster_cumulative,,,,,
1,33,22.61%,4.86%,4.95,20.04
2,928,12.73%,9.69%,1.38,2.61
0,6955,0.30%,5.15%,0.04,0.09
4,4571,-14.16%,10.91%,-1.27,-1.73
3,929,-46.13%,10.81%,-4.21,-5.16
inactive,97,nan%,nan%,nan,nan


## Phase 3: Baseline Backtesting

En esta fase establecemos el punto de referencia interno con estrategias clásicas. No buscamos una sola métrica, sino ver estabilidad entre validación y test, sensibilidad al factor de selección y qué optimizadores generalizan mejor. El resumen visual comparará `validation_sharpe` frente a `test_sharpe`, mostrará los mejores trials y resumirá qué optimizadores/factores dominan la frontera superior.

In [8]:
run_python(['scripts/run_phase3.py'])

01:30:32 | INFO     |     Sharpe: train=0.536, val=0.250, test=0.636
01:30:32 | INFO     | Saved trials\results.csv (378 trials)
01:30:32 | INFO     |   Completed in 59m 44s (memory: +79.1 MB)
01:30:32 | INFO     | 
--- Generate Reports ---
01:30:32 | INFO     | Report saved to PHASE3_SUMMARY.md
01:30:32 | INFO     | 
01:30:32 | INFO     | RESULTS SUMMARY
01:30:32 | INFO     | ============================================================
01:30:32 | INFO     | Best configuration:
01:30:32 | INFO     |   Optimizer: min_variance
01:30:32 | INFO     |   Factor: rolling_return_21d
01:30:32 | INFO     |   N Selected: 50
01:30:32 | INFO     |   Sharpe (val): 2.7915
01:30:32 | INFO     |   Sharpe (test): 1.8742
01:30:32 | INFO     | 
Top 5 by Validation Sharpe:
01:30:32 | INFO     |   min_variance|rolling_return_21d|50: val=2.792, test=1.874
01:30:32 | INFO     |   max_sharpe|rolling_return_21d|200: val=2.785, test=1.954
01:30:32 | INFO     |   min_variance|rolling_return_21d|200: val=2.381, te

In [9]:
phase3_dir = latest_run_dir(PROJECT_ROOT / 'outputs' / 'baselines')
trials = pd.read_csv(phase3_dir / 'trials' / 'results.csv')
trials = trials[trials['status'] == 'completed'].copy()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
for optimizer, grp in trials.groupby('optimizer'):
    axes[0].scatter(grp['sharpe_val'], grp['sharpe_test'], s=40, alpha=0.7, label=optimizer)
axes[0].axline((0, 0), slope=1, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Validation Sharpe vs Test Sharpe')
axes[0].set_xlabel('validation sharpe')
axes[0].set_ylabel('test sharpe')
axes[0].legend(loc='best', fontsize=10)

optimizer_summary = trials.groupby('optimizer').agg(
    mean_val_sharpe=('sharpe_val', 'mean'),
    mean_test_sharpe=('sharpe_test', 'mean'),
    mean_val_return=('return_ann_val', 'mean'),
    n_trials=('spec', 'count')
).sort_values('mean_test_sharpe', ascending=False)
axes[1].bar(optimizer_summary.index.astype(str), optimizer_summary['mean_test_sharpe'], color='#E45756')
axes[1].set_title('Mean Test Sharpe by Optimizer')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

top_trials = trials.sort_values(['sharpe_test', 'sharpe_val'], ascending=False)[[
    'optimizer', 'factor', 'n_select', 'lookback', 'sharpe_val', 'sharpe_test', 'information_ratio'
]].head(12)
display(top_trials)

display(optimizer_summary.style.format({
    'mean_val_sharpe': '{:.2f}',
    'mean_test_sharpe': '{:.2f}',
    'mean_val_return': '{:.2%}',
}))

,mean_val_sharpe,mean_test_sharpe,mean_val_return,n_trials
optimizer,,,,
momentum,0.40,0.76,1.99%,63
equal_weight,0.43,0.71,1.94%,63
vol_targeting,0.43,0.71,1.94%,63
risk_parity,0.12,0.43,1.27%,63
max_sharpe,0.05,0.35,1.29%,63
min_variance,0.08,0.32,1.38%,63


## Phase 4: Environment Validation

Antes de entrenar RL, validamos el simulador, el entorno Gym y la función de recompensa. Esta fase sirve para defender que el agente aprende sobre un entorno mecánicamente consistente: restricciones activas, pesos válidos, episodios ejecutables y rewards razonables. El resumen mostrará los tests de reward, la distribución de recompensas por episodio y los sanity checks clave.

In [10]:
run_python(['scripts/run_phase4.py'])

01:30:39 | INFO     |   Equal weight mean reward: -0.0136
01:30:39 | INFO     | Testing zero weights handling...
01:30:39 | INFO     | Simulator reset: 2020-06-07 00:00:00 to 2024-12-30 00:00:00, 1426 days, 318 rebalance dates
01:30:39 | INFO     |   Zero weights handled: True
01:30:39 | INFO     | Testing constraint enforcement...
01:30:39 | INFO     | Simulator reset: 2020-06-07 00:00:00 to 2024-12-30 00:00:00, 1426 days, 318 rebalance dates
01:30:39 | INFO     |   Max weight enforced: True (max=0.17)
01:30:39 | INFO     |   Completed in 201ms (memory: +1.0 MB)
01:30:39 | INFO     | Report saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\environment\20260328_013037_weekly_std\PHASE4_SUMMARY.md
01:30:40 | INFO     | 
01:30:40 | INFO     | PHASE COMPLETE: Phase 4: Environment Validation
01:30:40 | INFO     | ======================================================================
01:30:40 | INFO     | Total time: 2.4s
01:30:40 | INFO     | Memory delta: +275.8 MB
01

In [11]:
phase4_dir = latest_run_dir(PROJECT_ROOT / 'outputs' / 'environment')
phase4 = read_json(phase4_dir / 'phase4_results.json')
reward_tests = pd.Series(phase4.get('reward_tests', {}), name='score').sort_values(ascending=False)
episodes = pd.DataFrame(phase4.get('test_episodes', []))
sanity = pd.DataFrame([phase4.get('sanity_checks', {})])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
reward_tests.plot(kind='bar', ax=axes[0], color='#54A24B')
axes[0].set_title('Reward Function Smoke Tests')
axes[0].set_ylabel('score')
axes[0].tick_params(axis='x', rotation=35)

if not episodes.empty:
    axes[1].bar(episodes['episode'].astype(str), episodes['total_reward'], color='#B279A2')
    axes[1].set_title('Recompensa Total por Episodio')
    axes[1].set_xlabel('episode')
    axes[1].set_ylabel('total_reward')
else:
    axes[1].text(0.5, 0.5, 'No episode data', ha='center', va='center')
plt.tight_layout()
plt.show()

display(pd.DataFrame([{
    'n_episodes': phase4.get('n_episodes'),
    'n_algos': phase4.get('n_algos'),
    'n_days': phase4.get('n_days'),
    'rebalance_frequency': phase4.get('config', {}).get('rebalance_frequency'),
    'observation_shape': tuple(phase4.get('env', {}).get('observation_shape', [])),
    'action_shape': tuple(phase4.get('env', {}).get('action_shape', [])),
}]))
display(sanity)

,equal_weight_mean_reward,zero_weights_handled,max_weight_enforced
0,-0.013616,True,True


## Phase 5: RL Agent Training

Aquí entrenamos el agente `PPO` sobre el universo filtrado por los mejores clusters temporales de `Phase 2`, con `reward_type = pure_returns`, sin hybrid mode y usando GPU. La lectura para presentación debe enseñar dos ideas: qué universo se seleccionó realmente y cómo evolucionó el entrenamiento. El resumen visual mostrará la evolución temporal de métricas del entrenamiento y la tabla de clusters/algoritmos finalmente usados.

In [17]:
run_python([
    'scripts/run_phase5.py',
    '--agent', 'ppo',
    '--timesteps', '1M',
    '--gpu-env',
    '--max-resources',
    '--reward-type', 'pure_returns',
    '--no-hybrid',
    '--phase2-cluster-filter',
    '--phase2-analysis-dir', 'data/processed/analysis',
    '--phase2-cluster-source', 'temporal_cumulative',
    '--phase2-cluster-score-mode', 'return_low_vol',
    '--phase2-cluster-top-k', '2',
    '--phase2-cluster-min-size', '20',
    '--phase2-cluster-min-return', '0.01',
    '--phase2-cluster-max-vol', '0.12',
    '--holdout-start-date', '2024-01-01',
    '--holdout-end-date','2024-12-31'
])

13:53:07 | INFO     | Validation mean reward: -987.41
13:53:07 | INFO     | Validation mean return: 0.00%
13:53:07 | INFO     |   Completed in 17m 36s (memory: +664.0 MB)
13:53:07 | INFO     | 
--- 5.5 Generate Report ---
13:53:07 | INFO     | Report saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\rl_training\20260328_133529_ppo\PHASE5_SUMMARY.md
13:53:07 | INFO     | Run manifest: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\rl_training\20260328_133529_ppo\run_info.json
13:53:07 | INFO     |   Completed in 4ms (memory: +0.0 MB)
13:53:07 | INFO     | 
13:53:07 | INFO     | PHASE COMPLETE: Phase 5: RL Agent Training
13:53:07 | INFO     | ======================================================================
13:53:07 | INFO     | Total time: 17m 38s
13:53:07 | INFO     | Memory delta: +1167.8 MB
13:53:07 | INFO     | Steps: 6/6 completed
13:53:07 | INFO     | Metrics saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\rl_training\

In [18]:
run_python([
    'scripts/run_phase6.py',
    '--include-baselines',
])

13:53:16 | INFO     | 
13:53:16 | INFO     | PHASE COMPLETE: Phase 6: Walk-Forward Evaluation
13:53:16 | INFO     | ======================================================================
13:53:16 | INFO     | Total time: 4.1s
13:53:16 | INFO     | Memory delta: +662.4 MB
13:53:16 | INFO     | Steps: 11/11 completed
13:53:16 | INFO     | Metrics saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\evaluation\20260328_135312_ppo_latest\phase6_metrics.json
13:53:16 | INFO     | Results saved to: C:\Users\Admin\DataspellProjects\athenai_competition\outputs\evaluation\20260328_135312_ppo_latest\phase6_results.json
13:53:16 | INFO     | 
13:53:16 | INFO     | PHASE COMPLETE: Phase 6: Walk-Forward Evaluation
13:53:16 | INFO     | ======================================================================
13:53:16 | INFO     | Total time: 4.1s
13:53:16 | INFO     | Memory delta: +662.6 MB
13:53:16 | INFO     | Steps: 11/11 completed

PHASE SUMMARY: Phase 6: Walk-Forward Evaluation

In [19]:
run_python(['scripts/summarize_phase6_runs.py'])

[info] No --run-id/--prefix provided. Using latest Phase 6 run: 20260328_135312_ppo_latest
                    run_id agent  annualized_return  annualized_volatility  sharpe_ratio  max_drawdown  information_ratio  benchmark_return  benchmark_sharpe best_baseline  best_baseline_return  best_baseline_sharpe  return_vs_benchmark  sharpe_vs_benchmark  return_vs_best_baseline  sharpe_vs_best_baseline                                                                                                                       equity_plot
20260328_135312_ppo_latest   ppo           0.408442               0.047276      7.872106      -0.00407          11.691363           0.02621          0.962534  Min Variance              0.045699              3.375371             0.382232             6.909572                 0.362744                 4.496735 C:\Users\Admin\DataspellProjects\athenai_competition\outputs\evaluation\20260328_135312_ppo_latest\phase6_equity_vs_benchmark.png
